# 🔬 EDA Laboratory: `statsbomb_fa_women's_super_league_2019_2020_match_2275105.csv`
Execute the cells below to explore and clean the dataset. Modify parameters as needed.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
import sys
import warnings
warnings.filterwarnings('ignore')

# Append root directory to path so we can import local scripts
sys.path.append('../../') 
from scripts.db_uploader import upload_to_postgres

df = pd.read_csv('../../data/raw/statsbomb_fa_women's_super_league_2019_2020_match_2275105.csv', low_memory=False)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

In [ ]:
# 1. BASIC EXPLORATION
df.info()
print(df.isnull().sum())

In [ ]:
# 2. CATEGORICAL VARIABLES ENCODING (Label Encoding & Boolean Imputation)
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
for col in cat_cols:
    unique_vals = [str(x).lower() for x in df[col].dropna().unique()]
    if 'true' in unique_vals or 'false' in unique_vals:
        df[col] = df[col].fillna(False).map({True: 1, False: 0, 'True': 1, 'False': 0, '1': 1, '0': 0}).fillna(0).astype(int)
    else:
        df[col] = df[col].fillna('Unknown')
        df[col+'_encoded'] = LabelEncoder().fit_transform(df[col].astype(str))
df = df.dropna(how='all', axis=1)
print('Encoding and Imputation completed successfully.')

In [ ]:
# 3. CORRELATION MATRIX
numerical_data = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 8))
sns.heatmap(numerical_data.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 🚀 FINISH EDA: DATABASE UPLOAD
When you are satisfied with the data cleaning, execute this cell to upload the polished DataFrame to PostgreSQL (DBeaver).

In [ ]:
upload_to_postgres(df, table_name='statsbomb_fa_women's_super_league_2019_2020_match_2275105_clean')